# 01. First API Call to Azure OpenAI (Responses API)

This notebook is the very first hands-on script in the **AI-103 -> Section 01: Building the Foundation (Models, Tools, and the Developer)** track. It shows the smallest possible round-trip to an Azure OpenAI deployment using the **Responses API** -- the same official `openai` Python SDK you would use against OpenAI's public API, pointed at your Azure resource via a `base_url` and an API key.

**Difficulty: Beginner**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt` -- no extra installs needed):
- `openai` -- the official OpenAI Python SDK; Azure OpenAI's newer endpoints speak the same wire format, so the same SDK works against both OpenAI and Azure.
- `python-dotenv` -- loads secrets/config from the repo-root `.env` instead of hardcoding them.

**Azure resources**
- An **Azure OpenAI** resource (or an Azure AI Foundry resource with the OpenAI v1 API surface enabled) with a model **deployed** -- e.g. `gpt-4.1`. You need the resource's endpoint and, for this notebook specifically, an **API key**.

**Environment variables** -- add these to the root `.env` (see this repo's `CLAUDE.md`):
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/openai/v1
AZURE_OPENAI_API_KEY=<your-api-key>
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```
> Note: this repo's `.env` currently only ships `AZURE_AI_PROJECT_ENDPOINT` / `AZURE_AI_MODEL_DEPLOYMENT` (for the Agent-Service labs in `11_azure_ai_foundry/`). You'll need to add the three keys above yourself before this notebook can actually call Azure -- the code below is written to run end-to-end once they exist.


## What You'll Learn
- The shape of an Azure OpenAI **v1 Responses API** endpoint (`.../openai/v1`) and how it lets you use the stock `openai` SDK unmodified
- How to construct an `OpenAI` client pointed at Azure via `base_url` + `api_key`
- The **Responses API** call shape: `client.responses.create(model=..., input=...)`
- The `response.output_text` convenience property
- Why the **deployment name**, not the base model name, is what you pass as `model` in Azure OpenAI


### 1. Imports and configuration

We load the endpoint, API key, and deployment name from environment variables instead of hardcoding them (the original script had `api_key = "<API_KEY>"` as a literal placeholder -- never commit a real key like that). `load_dotenv()` reads the repo-root `.env` file so `os.getenv(...)` can see it.

**Exam tip:** AI-102 expects you to know Azure OpenAI's two auth models: **API key** (`api-key` header / `api_key=` in the SDK) and **Microsoft Entra ID** (OAuth bearer tokens via `DefaultAzureCredential`). Key-based auth is quick to start with but the key is a long-lived secret; Entra ID is the recommended pattern for production because it uses short-lived tokens and role-based access control (RBAC) instead of a shared secret.

**Alternatives:** Every other script in this section (`02` onward) swaps to Entra ID auth -- see the next notebook. You could adopt that here today by replacing the last two lines with:
```python
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
# then pass api_key=token_provider to OpenAI(...) instead of a raw key
```


In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
api_key = os.getenv("AZURE_OPENAI_API_KEY")


### 2. Build the client

`OpenAI(base_url=..., api_key=...)` is the entire integration surface -- no separate `AzureOpenAI` class is needed because the endpoint already exposes the OpenAI-compatible `/openai/v1` path. This is a relatively new option for Azure OpenAI (the older pattern used a dedicated `AzureOpenAI` client with an `api_version` query parameter).

**Exam tip:** Know both client shapes for the exam: `openai.AzureOpenAI(azure_endpoint=..., api_key=..., api_version=...)` (classic, versioned) vs `openai.OpenAI(base_url="<resource>/openai/v1", api_key=...)` (newer v1 preview surface, no `api_version` needed). Both ultimately hit the same underlying deployments.

**Alternatives:** `AzureOpenAI` from the `openai` package remains fully supported and is what most existing Azure OpenAI tutorials still show; pick it if you need explicit `api_version` pinning for a preview feature.


In [2]:
client = OpenAI(
    base_url=endpoint,
    api_key=api_key,
)


### 3. Send the request

`client.responses.create(model=deployment_name, input=...)` is the Responses API's basic single-turn call: `model` must be the **deployment name** you chose in Azure OpenAI Studio/AI Foundry (not necessarily the underlying base model's name), and `input` can be a plain string for simple prompts.

**Exam tip:** The Responses API is OpenAI/Azure's newer unified API (superseding the older Assistants API and complementing Chat Completions) -- it natively supports multi-turn conversation state, built-in tools (web search, code interpreter, file search), and reasoning models through one consistent call shape. AI-102 increasingly references it as the forward path for building agentic apps on Azure OpenAI.

**Alternatives:** `client.chat.completions.create(model=..., messages=[...])` is the classic Chat Completions call and still works identically against Azure OpenAI; pick it for compatibility with existing chat-message-based code.


In [3]:
response = client.responses.create(
    model=deployment_name,
    input="What are the three main benefits of using managed AI endpoints in the cloud?",
)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************2nEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

### 4. Read the answer

`response.output_text` is a convenience property that concatenates all text output segments from the response into a single string -- handy for simple cases where you don't need to inspect the full structured `response.output` list (we'll use that richer structure later, e.g. in the code-interpreter notebook).


In [ ]:
print(f"answer: {response.output_text}")


## Summary

You built an `OpenAI` SDK client pointed at an Azure OpenAI deployment via `base_url` + API key, sent one prompt through the Responses API, and printed `response.output_text`. This is the minimal building block every later notebook in this chapter extends -- with Entra ID auth, system instructions, multimodal input, and built-in tools.


## Try It Yourself
1. **Easy:** Change the `input` string to ask a different question and re-run.
2. **Medium:** Switch this notebook to Entra ID auth (see the Alternatives note above) instead of an API key -- no code outside the client-construction cell should need to change.
3. **Advanced:** Add error handling around `client.responses.create(...)` for `openai.AuthenticationError` and `openai.RateLimitError`, printing a friendly message instead of a raw traceback.
